# Unit 2: Linear Algebra & NumPy

Today's goal: get comfortable with NumPy. By the end of this notebook you should be able to:
1. Create, reshape, slice, and index arrays
2. Use vectorized operations instead of loops
3. Perform matrix operations (dot product, transpose, multiply, inverse)
4. Compute distance metrics between vectors
5. Connect it back to regression from Unit 1

In [1]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

---
# Part 1: Why NumPy?

Python lists are flexible but slow. NumPy arrays are fast. How much does it matter?

## 1.1 The race

Task: square every element in a million-element array. Three approaches.

In [ ]:
data = np.random.rand(1000000)

# --- For loop ---
start = time.time()
squared_loop = []
for x in data:
    squared_loop.append(x ** 2)
loop_time = time.time() - start
print(f"For loop:           {loop_time:.4f}s")



In [ ]:
# --- Pandas apply ---
df_tmp = pd.DataFrame(data, columns=['val'])
start = time.time()
squared_apply = df_tmp['val'].apply(lambda x: x ** 2)
apply_time = time.time() - start
print(f"Pandas apply:       {apply_time:.4f}s")



In [ ]:
# --- NumPy vectorized ---
start = time.time()
squared_np = np.square(data)
np_time = time.time() - start
print(f"NumPy vectorized:   {np_time:.4f}s")

print(f"\nNumPy is ~{loop_time / np_time:.0f}x faster than a for loop")

**Why is NumPy so much faster?**

*Your answer:*


## 1.2 Your turn

Create a 1000 x 1000 random matrix and square every element using all three approaches. Benchmark each.

In [ ]:
# Your code

---
# Part 2: NumPy Fundamentals

## 2.1 Creating arrays

In [ ]:
# From a list
v = np.array([1, 2, 3, 4, 5])
print("v:", v)
print("shape:", v.shape)
print("dtype:", v.dtype)

# 2D array
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
print("\nA:\n", A)
print("shape:", A.shape)

In [ ]:
# Handy constructors
print("zeros:\n", np.zeros((2, 3)))
print("\nones:\n", np.ones((2, 3)))
print("\nidentity:\n", np.eye(3))
print("\narange:", np.arange(0, 10, 2))
print("\nlinspace:", np.linspace(0, 1, 5))

## 2.2 Reshaping

In [ ]:
arr = np.arange(1, 13)
print("Original:", arr, "| shape:", arr.shape)

reshaped = arr.reshape(3, 4)
print("\nReshaped to 3x4:\n", reshaped)

print("\nReshaped to 4x3:\n", arr.reshape(4, 3))

# -1 lets NumPy figure out one dimension
print("\nReshape with -1:\n", arr.reshape(2, -1))

**Predict:** What shape is `np.arange(24).reshape(2, 3, -1)`?

*Your answer:*


In [ ]:
# Verify
np.arange(24).reshape(2, 3, -1).shape

## 2.2b Stacking and squeezing

`hstack` and `vstack` combine arrays horizontally and vertically. `squeeze` removes dimensions of size 1.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# hstack: combine side by side (horizontally)
print("hstack:", np.hstack([a, b]))

# vstack: stack on top of each other (vertically)
print("vstack:\n", np.vstack([a, b]))

In [ ]:
# squeeze: removes dimensions of size 1
c = np.array([[[1, 2, 3]]])  # shape (1, 1, 3)
print("Before squeeze:", c.shape)
print("After squeeze: ", c.squeeze().shape)
print("Values:", c.squeeze())

**Your turn:** Given these two arrays, use `vstack` to create a 4x3 matrix, then use `hstack` to create a 2x6 matrix.

```python
X = np.array([[1, 2, 3],
              [4, 5, 6]])
Y = np.array([[7, 8, 9],
              [10, 11, 12]])
```

In [ ]:
# Your code

## 2.3 Indexing and slicing

In [ ]:
M = np.arange(12).reshape(3, 4)
print("M:\n", M)
print("\nRow 0:", M[0])
print("Col 1:", M[:, 1])
print("Element (1,2):", M[1, 2])
print("Rows 0-1, Cols 2-3:\n", M[:2, 2:])

**Your turn:** From the matrix `M` above, extract:
1. The last row
2. The last column
3. The 2x2 submatrix in the bottom-right corner

In [ ]:
# Your code

## 2.4 Boolean masking

One of the most useful NumPy patterns: filter arrays using conditions.

In [ ]:
data = np.array([3, 7, 1, 9, 4, 6, 2, 8])

# The condition creates a boolean array
mask = data > 5
print("data:", data)
print("mask:", mask)
print("filtered:", data[mask])

**Your turn:** Given the array below, use boolean masking to:
1. Get all even numbers
2. Get all numbers between 10 and 20 (inclusive)
3. Replace all values greater than 15 with -1

In [ ]:
arr = np.array([2, 17, 8, 23, 11, 14, 6, 19, 25, 10])

# Your code

## 2.5 Broadcasting

NumPy can operate on arrays of different shapes by "stretching" the smaller one.

In [ ]:
# Scalar broadcast
a = np.array([1, 2, 3])
print("a * 10:", a * 10)

# Row + column broadcast
row = np.array([1, 2, 3])
col = np.array([[10], [20], [30]])
print("\nrow + col:\n", row + col)

## 2.6 Data types

In [2]:
int_array = np.array([1, 0, 3, 5])
print("int:", int_array, "| dtype:", int_array.dtype)
print("float:", int_array.astype(float))
print("bool:", int_array.astype(bool))  # 0 -> False, nonzero -> True

# Memory matters at scale
big = np.random.rand(1000000)
print(f"\nfloat64: {big.nbytes / 1e6:.1f} MB")
print(f"float32: {big.astype(np.float32).nbytes / 1e6:.1f} MB")

int: [1 0 3 5] | dtype: int64
float: [1. 0. 3. 5.]
bool: [ True False  True  True]

float64: 8.0 MB
float32: 4.0 MB


---
# Part 3: Matrix Operations

## 3.1 Transpose

Flips rows and columns. If $A$ is $m \times n$, then $A^T$ is $n \times m$.

In [ ]:
B = np.arange(1, 13).reshape(3, 4)
print("B (3x4):\n", B)
print("\nB^T (4x3):\n", B.T)

## 3.2 Dot product

Two vectors: $a \cdot b = \sum_i a_i b_i$

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("np.dot(a, b):", np.dot(a, b))
print("By hand: 1*4 + 2*5 + 3*6 =", 1*4 + 2*5 + 3*6)

## 3.3 Matrix multiplication

Each element in the result is a dot product of a row from A and a column from B.

**Shape rule:** $(m \times \mathbf{n}) \cdot (\mathbf{n} \times p) = (m \times p)$ — inner dimensions must match.

In [ ]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

print("A @ B:\n", A @ B)

**Predict:** What is element (0, 1) of `A @ B`? Work it out by hand, then verify.

*Your answer:*


**Your turn:** For each pair, predict whether the multiplication works and the output shape. Then verify.

| A shape | B shape | Works? | Result shape |
|---------|---------|--------|--------------|
| (3, 4) | (4, 2) | ? | ? |
| (3, 4) | (3, 2) | ? | ? |
| (5, 3) | (3, 1) | ? | ? |
| (2, 2) | (2, 2) | ? | ? |

In [ ]:
# Your code — try each

## 3.4 Inverse

$A A^{-1} = I$ (the identity matrix). Only square matrices can have an inverse, and not all do.

In [ ]:
A = np.array([[2, 1],
              [5, 3]])

A_inv = np.linalg.inv(A)
print("A:\n", A)
print("\nA inverse:\n", A_inv)
print("\nA @ A_inv:\n", np.round(A @ A_inv, 10))

---
# Part 4: Distances

Many algorithms (kNN, clustering, embeddings) rely on measuring distance between points. Let's implement three common ones using only NumPy.

In [ ]:
point1 = np.array([1, 2])
point2 = np.array([4, 6])

**Your turn:** Write functions for each. No built-in distance functions — use the NumPy operations you've learned.

1. **Euclidean distance**: $d = \sqrt{\sum_i (a_i - b_i)^2}$
2. **Manhattan distance**: $d = \sum_i |a_i - b_i|$
3. **Cosine similarity**: $\cos(\theta) = \frac{a \cdot b}{\|a\| \|b\|}$

In [ ]:
# Your code: Euclidean

In [ ]:
# Your code: Manhattan

In [ ]:
# Your code: Cosine similarity

**Quick check:** Cosine similarity is 1 when vectors point the same direction, 0 when perpendicular, -1 when opposite. What's the cosine similarity of `[1, 0]` and `[0, 1]`? Verify with your function.

In [ ]:
# Your code

---
# Part 5: Connecting to Regression

In Unit 1 you used `statsmodels` to fit regressions. Now let's see that regression is just the NumPy operations you learned today.

## 5.1 Generate data

We'll simulate data where we know the true answer: $y = 5 + 3x + \epsilon$, where $\epsilon \sim N(0, 10)$.

In [ ]:
np.random.seed(42)
n = 500

beta_0_true = 5
beta_1_true = 3

x = np.random.rand(n) * 10
noise = np.random.normal(0, 10, n)
y = beta_0_true + beta_1_true * x + noise

plt.scatter(x, y, alpha=0.3, s=10)
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Simulated: y = {beta_0_true} + {beta_1_true}x + noise')
plt.tight_layout()

## 5.2 Compute betas with NumPy

You have these formulas:

$$\hat{\beta}_1 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n} (x_i - \bar{x})^2}$$

$$\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$$

**Your turn:** Compute $\hat{\beta}_0$ and $\hat{\beta}_1$ using NumPy operations (no loops). Print them.

In [ ]:
# Your code

## 5.3 Compare with statsmodels

Now fit the same model with `sm.OLS` like you did in Unit 1. Do your betas match?

In [ ]:
# Your code

## 5.4 Sample size and estimation

**Your turn:** 
1. Write a function `estimate_beta1(n)` that generates data from $y = 5 + 3x + \epsilon$, computes $\hat{\beta}_1$, and returns it.
2. For sample sizes `[20, 50, 100, 500, 1000, 5000]`, run the function 100 times each.
3. Plot all estimates as a scatter: x-axis is sample size, y-axis is $\hat{\beta}_1$. Add a horizontal line at the true value (3).

What do you observe?

In [ ]:
# Your code

*Your interpretation:*
